In [20]:
# Filemame......: 44 Securing Infrastructure for AI Applications
# Language......: Python
# Tools.......... Visual Source Code (VSC)
# Class.........: ITAI 1372 Artificial Intelligence in Cybersecurity
# Semster.......: Fall 2026
# Class Type....: Online
# Instructor....: Hien Nguyen
# Student.......: Olis Bahari
# Version.......: V1.0

In [21]:
import google.auth

credentials, project_id = google.auth.default()

print("Authentication successful!")
print("Project ID:", project_id)
print("Quota Project:", credentials.quota_project_id)

Authentication successful!
Project ID: dynamic-circlet-j41fc
Quota Project: dynamic-circlet-j41fc


In [22]:
LOCATION = "global"
UNSTRUCTURED_TEMPLATE_ID = "deid-unstructured"
STRUCTURED_TEMPLATE_ID = "deid-structured"
IMAGE_TEMPLATE_ID = "deid-image"

In [23]:
# Google Cloud DLP template IDs

INSPECT_TEMPLATE_ID = "pii-finder"
UNSTRUCTURED_TEMPLATE_ID = "de-identify-unstructured"
CLOUDSDK_CORE_PROJECT="dynamic-circlet-j41fc"
TEST_STRING = (
    "My name is John Smith. "
    "My email is john.smith@example.com. "
    "My phone number is 415-555-1234."
)

print(TEST_STRING)

My name is John Smith. My email is john.smith@example.com. My phone number is 415-555-1234.


In [24]:
import os
import google.auth

credentials, project_id = google.auth.default()

PROJECT_ID = os.environ.get(
    "GOOGLE_CLOUD_PROJECT",
    project_id
)

print("Project ID:", PROJECT_ID)


print("Windows user:", os.environ.get("USERNAME"))

credentials, project_id = google.auth.default()

print("Authentication successful!")
print("Project ID:", project_id)
print("Credentials type:", type(credentials).__name__)


Project ID: dynamic-circlet-j41fc
Windows user: Lenovo
Authentication successful!
Project ID: dynamic-circlet-j41fc
Credentials type: Credentials


In [25]:
from google.cloud import dlp_v2

dlp = dlp_v2.DlpServiceClient()

parent = f"projects/{PROJECT_ID}/locations/global"

response = dlp.list_deidentify_templates(
    request={"parent": parent}
)

templates = list(response.deidentify_templates)

for template in templates:
    print("Template:", template.name)

if not templates:
    print("No de-identification templates found.")

Template: projects/dynamic-circlet-j41fc/locations/global/deidentifyTemplates/de-identify-image
Template: projects/dynamic-circlet-j41fc/locations/global/deidentifyTemplates/de-identify-structured
Template: projects/dynamic-circlet-j41fc/locations/global/deidentifyTemplates/de-identify-unstructured


In [26]:
def parent(project):
    return f"projects/{project}/locations/global"

def inspect_template_name(project):
    return (
        f"{parent(project)}/inspectTemplates/"
        f"{INSPECT_TEMPLATE_ID}"
    )

def deid_template_name(project, template_id):
    return (
        f"{parent(project)}/deidentifyTemplates/"
        f"{template_id}"
    )

In [27]:
from google.cloud import dlp_v2

dlp = dlp_v2.DlpServiceClient()

print("Google Cloud DLP client initialized successfully!")

Google Cloud DLP client initialized successfully!


In [28]:
!python dlp_pipeline.py all --project dynamic-circlet-j41fc

  File "c:\Dropbox\Artifical Intellegence\Houston City College (HCC)\04 Artificial Intelligence in Cybersecurity\Olis-Bahari-CS-ITAI-1372\Lab\41 Securing Data Used for AI Applications\dlp_pipeline.py", line 2
    %pip install --quiet google-cloud-dlp google-cloud-storage
    ^
SyntaxError: invalid syntax


In [29]:
import argparse
import os
import subprocess
import sys
import time

from google.cloud import dlp_v2
from google.cloud import storage

In [30]:
%pip install google-cloud-dlp
from google.cloud import dlp_v2
print("OK:", dlp_v2.InfoType)

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
OK: <class 'google.cloud.dlp_v2.types.storage.InfoType'>


In [31]:
import os

project_id = os.environ.get('GOOGLE_CLOUD_PROJECT')

if not project_id:
    print("❌ GOOGLE_CLOUD_PROJECT environment variable is NOT set.")
elif project_id == 'your-gcp-project-id':
    print("⚠️ GOOGLE_CLOUD_PROJECT is still set to the default placeholder: 'your-gcp-project-id'.")
    print("Please edit cell 8c459dc1 with your actual GCP Project ID and re-run it.")
else:
    print(f"✅ GOOGLE_CLOUD_PROJECT is successfully configured to: '{project_id}'")

✅ GOOGLE_CLOUD_PROJECT is successfully configured to: 'dynamic-circlet-j41fc'


In [32]:
from __future__ import annotations
import argparse, os, subprocess, sys, time

from google.api_core import exceptions as gexc
from google.cloud import dlp_v2, storage

LOCATION = "global"
INSPECT_TEMPLATE_ID = "pii-finder"
UNSTRUCTURED_TEMPLATE_ID = "de-identify-unstructured"
STRUCTURED_TEMPLATE_ID = "de-identify-structured"
IMAGE_TEMPLATE_ID = "de-identify-image"
JOB_ID = "pii-remover"

INFO_TYPES = [
    "CREDIT_CARD_EXPIRATION_DATE", "CREDIT_CARD_NUMBER", "DATE_OF_BIRTH",
    "DRIVERS_LICENSE_NUMBER", "EMAIL_ADDRESS", "GCP_API_KEY", "GCP_CREDENTIALS",
    "ORGANIZATION_NAME", "PASSWORD", "PERSON_NAME", "PHONE_NUMBER",
    "US_SOCIAL_SECURITY_NUMBER",
]
STRUCTURED_FIELDS = ["user_id", "message"]
TEST_STRING = ("Hi, my name is Alex and my SSN is 555-11-5555. "
               "You can reach me at +1-555-555-5555.")

In [33]:
%%writefile dlp_pipeline.py
# Install necessary Google Cloud SDK libraries for Colab environment
%pip install --quiet google-cloud-dlp google-cloud-storage

#!/usr/bin/env python3
"""
Securing Data Used for AI Applications — automated PII sanitization pipeline.

Builds the same resources as the Google Codelab, but in code, using the
Sensitive Data Protection (Cloud DLP) API:

  1. Inspect template          pii-finder                 (what counts as sensitive)
  2. De-identify templates     de-identify-unstructured   (text/logs -> [INFO_TYPE])
                               de-identify-structured     (CSV user_id/message -> masked with #)
                               de-identify-image          (redaction boxes over PII in images)
  3. Inspection job            pii-remover                (scans gs://input-<project>, writes
                                                           de-identified copies to gs://output-<project>)
  4. Verification              before/after comparison of input vs. output files

Usage (run in Cloud Shell after ./setup.sh):
  python3 dlp_pipeline.py templates     # create / replace the 4 templates
  python3 dlp_pipeline.py test          # quick API sanity test on a sample string
  python3 dlp_pipeline.py job           # create the pii-remover job
  python3 dlp_pipeline.py wait          # poll the job until DONE, print findings
  python3 dlp_pipeline.py verify        # print before/after for text + CSV, list images
  python3 dlp_pipeline.py all           # templates -> test -> job -> wait -> verify
  python3 dlp_pipeline.py cleanup       # delete job + templates (buckets: see cleanup.sh)

Project is taken from --project, $GOOGLE_CLOUD_PROJECT, or `gcloud config`.
"""

from __future__ import annotations

import argparse
import os
import subprocess
import sys
import time

from google.api_core import exceptions as gexc
from google.cloud import dlp_v2, storage

LOCATION = "global"

INSPECT_TEMPLATE_ID = "pii-finder"
UNSTRUCTURED_TEMPLATE_ID = "de-identify-unstructured"
STRUCTURED_TEMPLATE_ID = "de-identify-structured"
IMAGE_TEMPLATE_ID = "de-identify-image"
JOB_ID = "pii-remover"

# infoTypes from the codelab's inspect template.
INFO_TYPES = [
    "CREDIT_CARD_EXPIRATION_DATE",
    "CREDIT_CARD_NUMBER",
    "DATE_OF_BIRTH",
    "DRIVERS_LICENSE_NUMBER",
    "EMAIL_ADDRESS",
    "GCP_API_KEY",
    "GCP_CREDENTIALS",
    "ORGANIZATION_NAME",
    "PASSWORD",
    "PERSON_NAME",
    "PHONE_NUMBER",
    "US_SOCIAL_SECURITY_NUMBER",
 ]

# CSV columns to sanitize. agent_id (employee address) is intentionally left alone.
STRUCTURED_FIELDS = ["user_id", "message"]

TEST_STRING = (
    "Hi, my name is Alex and my SSN is 555-11-5555. "
    "You can reach me at +1-555-555-5555."
)




Overwriting dlp_pipeline.py


In [34]:
# --------------------------------------------------------------------------- #
# Helpers
# --------------------------------------------------------------------------- #
def resolve_project(cli_value: str | None) -> str:
    project = cli_value or os.environ.get("GOOGLE_CLOUD_PROJECT")
    if not project:
        try:
            project = subprocess.check_output(
                ["gcloud", "config", "get-value", "project"],
                text=True, stderr=subprocess.DEVNULL,
            ).strip()
        except (OSError, subprocess.CalledProcessError):
            project = ""
    if not project:
        sys.exit("No project set. Use --project or `gcloud config set project <id>`.")
    return project


def parent(project: str) -> str:
    return f"projects/{project}/locations/{LOCATION}"


def inspect_template_name(project: str) -> str:
    return f"{parent(project)}/inspectTemplates/{INSPECT_TEMPLATE_ID}"


def deid_template_name(project: str, template_id: str) -> str:
    return f"{parent(project)}/deidentifyTemplates/{template_id}"


def info_types() -> list[dlp_v2.InfoType]:
    return [dlp_v2.InfoType(name=n) for n in INFO_TYPES]




In [35]:
def cmd_templates(dlp: dlp_v2.DlpServiceClient, project: str) -> None:
    """Create all templates; if one already exists, replace it so reruns are idempotent."""
    name = inspect_template_name(project)
    tmpl = dlp_v2.InspectTemplate(display_name="PII finder",
                                  inspect_config=build_inspect_config())
    try:
        dlp.create_inspect_template(request={
            "parent": parent(project),
            "inspect_template": tmpl,
            "template_id": INSPECT_TEMPLATE_ID,
        })
        print(f"created  {name}")
    except gexc.AlreadyExists:
        dlp.update_inspect_template(name=name, inspect_template=tmpl)
        print(f"updated  {name}")

    for template_id, (display, builder) in DEID_TEMPLATES.items():
        name = deid_template_name(project, template_id)
        tmpl = dlp_v2.DeidentifyTemplate(display_name=display,
                                         deidentify_config=builder())
        try:
            dlp.create_deidentify_template(request={
                "parent": parent(project),
                "deidentify_template": tmpl,
                "template_id": template_id,
            })
            print(f"created  {name}")
        except gexc.AlreadyExists:
            dlp.update_deidentify_template(name=name, deidentify_template=tmpl)
            print(f"updated  {name}")

In [36]:
# --------------------------------------------------------------------------- #
# Template definitions
# --------------------------------------------------------------------------- #
def build_inspect_config() -> dlp_v2.InspectConfig:
    return dlp_v2.InspectConfig(
        info_types=info_types(),
        min_likelihood=dlp_v2.Likelihood.POSSIBLE,
        include_quote=True,
    )


def build_unstructured_config() -> dlp_v2.DeidentifyConfig:
    """Free text / logs: replace each finding with its infoType name,
    e.g. 'John Doe' -> '[PERSON_NAME]'. Keeps sentence context for the model."""
    return dlp_v2.DeidentifyConfig(
        info_type_transformations=dlp_v2.InfoTypeTransformations(
            transformations=[
                dlp_v2.InfoTypeTransformations.InfoTypeTransformation(
                    primitive_transformation=dlp_v2.PrimitiveTransformation(
                        replace_with_info_type_config=dlp_v2.ReplaceWithInfoTypeConfig()
                    )
                )
            ]
        )
    )


def build_structured_config() -> dlp_v2.DeidentifyConfig:
    """CSV / tables: in user_id and message only, mask findings with '#',
    ignoring punctuation so the shape survives (user26@example.org ->
    ######@#######.###). Other columns (agent_id, dates, ids) keep their utility."""
    mask = dlp_v2.PrimitiveTransformation(
        character_mask_config=dlp_v2.CharacterMaskConfig(
            masking_character="#",
            characters_to_ignore=[
                dlp_v2.CharsToIgnore(
                    common_characters_to_ignore=dlp_v2.CharsToIgnore.CommonCharsToIgnore.PUNCTUATION
                )
            ],
        )
    )
    return dlp_v2.DeidentifyConfig(
        record_transformations=dlp_v2.RecordTransformations(
            field_transformations=[
                dlp_v2.FieldTransformation(
                    fields=[dlp_v2.FieldId(name=f) for f in STRUCTURED_FIELDS],
                    info_type_transformations=dlp_v2.InfoTypeTransformations(
                        transformations=[
                            dlp_v2.InfoTypeTransformations.InfoTypeTransformation(
                                primitive_transformation=mask
                            )
                        ]
                    ),
                )
            ]
        )
    )


def build_image_config() -> dlp_v2.DeidentifyConfig:
    """Images: draw black boxes over every detected infoType from pii-finder."""
    return dlp_v2.DeidentifyConfig(
        image_transformations=dlp_v2.ImageTransformations(
            transforms=[
                dlp_v2.ImageTransformations.ImageTransformation(
                    all_info_types=dlp_v2.ImageTransformations.ImageTransformation.AllInfoTypes(),
                    redaction_color=dlp_v2.Color(red=0.0, green=0.0, blue=0.0),
                )
            ]
        )
    )


DEID_TEMPLATES = {
    UNSTRUCTURED_TEMPLATE_ID: ("De-identify unstructured text", build_unstructured_config),
    STRUCTURED_TEMPLATE_ID: ("De-identify structured CSV", build_structured_config),
    IMAGE_TEMPLATE_ID: ("De-identify images", build_image_config),
}




In [37]:
def _already_exists(err: Exception) -> bool:
    """DLP reports a taken template ID as InvalidArgument, not AlreadyExists."""
    return isinstance(err, gexc.AlreadyExists) or (
        isinstance(err, gexc.InvalidArgument) and "already in use" in str(err)
    )


def cmd_templates(dlp: dlp_v2.DlpServiceClient, project: str) -> None:
    """Create all templates; if one already exists, replace it so reruns are idempotent."""
    name = inspect_template_name(project)
    tmpl = dlp_v2.InspectTemplate(display_name="PII finder",
                                  inspect_config=build_inspect_config())
    try:
        dlp.create_inspect_template(request={
            "parent": parent(project),
            "inspect_template": tmpl,
            "template_id": INSPECT_TEMPLATE_ID,
        })
        print(f"created  {name}")
    except (gexc.AlreadyExists, gexc.InvalidArgument) as err:
        if not _already_exists(err):
            raise
        dlp.update_inspect_template(name=name, inspect_template=tmpl)
        print(f"updated  {name}")

    for template_id, (display, builder) in DEID_TEMPLATES.items():
        name = deid_template_name(project, template_id)
        tmpl = dlp_v2.DeidentifyTemplate(display_name=display,
                                         deidentify_config=builder())
        try:
            dlp.create_deidentify_template(request={
                "parent": parent(project),
                "deidentify_template": tmpl,
                "template_id": template_id,
            })
            print(f"created  {name}")
        except (gexc.AlreadyExists, gexc.InvalidArgument) as err:
            if not _already_exists(err):
                raise
            dlp.update_deidentify_template(name=name, deidentify_template=tmpl)
            print(f"updated  {name}")

In [38]:
from google.cloud import dlp_v2

TEST_STRING = (
    "My name is John Smith. "
    "My email is john.smith@example.com "
    "and my phone number is 415-555-2671."
)

def cmd_test(dlp: dlp_v2.DlpServiceClient, project: str):
    """Test Google Cloud DLP PII detection."""

    response = dlp.inspect_content(
        request={
            "parent": f"projects/{project}/locations/global",
            "inspect_config": {
                "info_types": [
                    {"name": "EMAIL_ADDRESS"},
                    {"name": "PHONE_NUMBER"},
                    {"name": "PERSON_NAME"},
                ],
                "include_quote": True,
            },
            "item": {
                "value": TEST_STRING
            },
        }
    )

    print("DLP Inspection Results:")

    for finding in response.result.findings:
        print("Type:", finding.info_type.name)
        print("Value:", finding.quote)
        print("----------------------")

In [39]:
# --------------------------------------------------------------------------- #
def main() -> None:
    os.environ["GOOGLE_CLOUD_PROJECT"] = "dynamic-circlet-j41fc"
    os.environ["GCLOUD_PROJECT"] = "dynamic-circlet-j41fc"
    print("Google Cloud Project:", os.environ["GOOGLE_CLOUD_PROJECT"])

    ap = argparse.ArgumentParser(description=__doc__,
                                 formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("command", choices=["templates", "test", "job", "wait",
                                        "verify", "all", "cleanup"])
    ap.add_argument("--project", help="GCP project ID (default: gcloud config)")
    ap.add_argument("--samples", type=int, default=1, help="files per type to show in verify")
    ap.add_argument("--job-only", action="store_true", help="cleanup: delete only the job")

    # Running inside Jupyter / VS Code / Colab: pick the command here
    if len(sys.argv) > 0 and "kernel" in sys.argv[0]:
        NOTEBOOK_COMMAND = "test"   # then change to "test", or use "all"
        print(f"[Notebook Mode]: running '{NOTEBOOK_COMMAND}'")
        args = ap.parse_args([NOTEBOOK_COMMAND])
    else:
        args = ap.parse_args()

    project = resolve_project(args.project)
    print(f"project: {project}\n")

    if args.command == "verify":
        cmd_verify(project, args.samples)
        return

    dlp = dlp_v2.DlpServiceClient()
    if args.command == "templates":
        cmd_templates(dlp, project)
    elif args.command == "test":
        cmd_test(dlp, project)
    elif args.command == "job":
        cmd_job(dlp, project)
    elif args.command == "wait":
        cmd_wait(dlp, project)
    elif args.command == "cleanup":
        cmd_cleanup(dlp, project, args.job_only)
    elif args.command == "all":
        cmd_templates(dlp, project); print()
        cmd_test(dlp, project); print()
        cmd_job(dlp, project); print()
        cmd_wait(dlp, project); print()
        cmd_verify(project, args.samples)


if __name__ == "__main__":
    main()

Google Cloud Project: dynamic-circlet-j41fc
[Notebook Mode]: running 'test'
project: dynamic-circlet-j41fc

DLP Inspection Results:
Type: PERSON_NAME
Value: John Smith
----------------------
Type: PERSON_NAME
Value: john
----------------------
Type: PERSON_NAME
Value: smith
----------------------
Type: EMAIL_ADDRESS
Value: john.smith@example.com
----------------------
Type: PHONE_NUMBER
Value: 415-555-2671
----------------------


In [40]:
# 1. Programmatically overwrite the global variables in our Colab notebook memory
import sys
import os

active_project = "dynamic-circlet-j41fc"
for name in ['PROJECT_ID', 'project_id']:
    globals()[name] = active_project
    setattr(sys.modules[__name__], name, active_project)

# 2. Update environment variables to direct any API clients and CLI commands
os.environ["GOOGLE_CLOUD_PROJECT"] = active_project
os.environ["CLOUDSDK_CORE_PROJECT"] = active_project

# 3. Clean up the local config and set project with explicit silent override flags to prevent prompts
!gcloud config set core/disable_prompts True --quiet
!gcloud config set project dynamic-circlet-j41fc --quiet

# 4. Enable the Sensitive Data Protection (DLP) API cleanly on our active project
!gcloud services enable dlp.googleapis.com --project=dynamic-circlet-j41fc --quiet

Updated property [core/disable_prompts].
[environment: untagged] Read more to tag: g.co/cloud/project-env-tag.
Updated property [core/project].
